<a href="https://colab.research.google.com/github/rafiebazine/rag-image-captioning/blob/hadil/phase_02_preprocessing/04_faiss_index.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip -q install faiss-cpu

import os, json, time, numpy as np, faiss
from google.colab import drive
drive.mount("/content/drive")

REPO = "/content/rag-image-captioning"
if not os.path.exists(REPO):
    os.system(f"git clone -q https://github.com/rafiebazine/rag-image-captioning.git {REPO}")
    os.system(f"cd {REPO} && git checkout -q hadil")

DRIVE_DIR = "/content/drive/MyDrive/rag_embeddings"
PROC = f"{REPO}/phase_02_preprocessing/processed"
IDX_DIR = f"{DRIVE_DIR}/faiss"
os.makedirs(IDX_DIR, exist_ok=True)
print(os.listdir(PROC))

Mounted at /content/drive
['metadata_kb_10k.json', 'kb_10k_clean.json', 'diversity_analysis.png', 'metadata_test_100.json', 'test_1000_clean.json', 'embedding_info.json', 'test_100_clean.json', 'metadata_test_1000.json', 'metadata_val_100.json', 'val_100_clean.json']


In [2]:
# 1) Load embeddings + mappings
img_emb = np.load(f"{DRIVE_DIR}/kb_10k_image_emb.npy")
img_ids = json.load(open(f"{DRIVE_DIR}/kb_10k_image_ids.json"))
txt_emb = np.load(f"{DRIVE_DIR}/kb_text_emb.npy")
owners  = json.load(open(f"{DRIVE_DIR}/kb_text_owners.json"))   # row i -> [image_id, caption_idx]
D = img_emb.shape[1]

# 2) Build indexes (exact search; vectors L2-normalised => inner product = cosine)
t0 = time.time(); index_img = faiss.IndexFlatIP(D); index_img.add(img_emb); t_img = time.time() - t0
t0 = time.time(); index_txt = faiss.IndexFlatIP(D); index_txt.add(txt_emb); t_txt = time.time() - t0
faiss.write_index(index_img, f"{IDX_DIR}/kb_image.index")
faiss.write_index(index_txt, f"{IDX_DIR}/kb_text.index")
print("image index:", index_img.ntotal, "| text index:", index_txt.ntotal)

# 3) Link vectors -> image ID, captions, metadata
kb = json.load(open(f"{PROC}/kb_10k_clean.json"))
meta = {m["image_id"]: m for m in json.load(open(f"{PROC}/metadata_kb_10k.json"))}
records = {}
for x in kb:
    m = meta[x["image_id"]]
    records[x["image_id"]] = {"file_name": x["file_name"], "captions_raw": x["captions_raw"],
        "captions_clean": x["captions_clean"], "object_counts": m["object_counts"],
        "num_objects": m["num_objects"], "supercategories": m["supercategories"]}
json.dump(records, open(f"{DRIVE_DIR}/kb_records.json", "w"))
assert set(records) == set(img_ids) and len(owners) == index_txt.ntotal
print("records linked:", len(records))

# 4) Checks
sims, idx = index_img.search(img_emb[:1000], 1)
print("image self-retrieval top-1 (should be 1.0):", float((idx[:, 0] == np.arange(1000)).mean()))

for n in ["val_100", "test_100", "test_1000"]:
    assert not (set(json.load(open(f"{DRIVE_DIR}/{n}_image_ids.json"))) & set(img_ids)), n
print("leakage: OK")

# 5) Retrieval latency (single query, ms)
val_emb = np.load(f"{DRIVE_DIR}/val_100_image_emb.npy")
def latency(index, q, k=10):
    ts = []
    for v in q:
        t = time.perf_counter(); index.search(v[None], k); ts.append((time.perf_counter() - t) * 1000)
    return round(float(np.mean(ts)), 3)
lat_img, lat_txt = latency(index_img, val_emb), latency(index_txt, val_emb)
print("latency ms/query  image->image:", lat_img, "| image->text:", lat_txt)

# 6) Example: val image -> top-3 captions
val = json.load(open(f"{PROC}/val_100_clean.json"))
s, i = index_txt.search(val_emb[:2], 3)
for q in range(2):
    print("\nGT:", val[q]["captions_raw"][0].strip())
    for r in range(3):
        iid, cidx = owners[i[q, r]]
        print(f"  {s[q, r]:.2f}", records[iid]["captions_raw"][cidx].strip())

# 7) Documentation
info = {"library": f"faiss {faiss.__version__}", "index_type": "IndexFlatIP (exact search)",
        "similarity_metric": "inner product on L2-normalised vectors (= cosine)",
        "dimension": D, "image_index_vectors": index_img.ntotal, "text_index_vectors": index_txt.ntotal,
        "image_index_size_mb": round(os.path.getsize(f"{IDX_DIR}/kb_image.index") / 1e6, 1),
        "text_index_size_mb": round(os.path.getsize(f"{IDX_DIR}/kb_text.index") / 1e6, 1),
        "indexing_time_sec": {"image": round(t_img, 3), "text": round(t_txt, 3)},
        "retrieval_latency_ms_per_query": {"image_to_image": lat_img, "image_to_text": lat_txt},
        "id_mapping": "image index row i -> kb_10k_image_ids.json[i]; text index row i -> kb_text_owners.json[i] = [image_id, caption_idx]"}
json.dump(info, open(f"{PROC}/index_info.json", "w"), indent=1)
print("\n", json.dumps(info, indent=1))

from google.colab import files
files.download(f"{PROC}/index_info.json")

image index: 10000 | text index: 50000
records linked: 10000
image self-retrieval top-1 (should be 1.0): 1.0
leakage: OK
latency ms/query  image->image: 2.733 | image->text: 15.191

GT: It looks like there is writing on the bottom of the Stop sign.
  0.34 A stop sign that has been graffitied to say never stop loving.
  0.33 A STOP sign with the word IT written in graffiti after STOP
  0.33 Closeup view of a stop sign with a black background.

GT: A city street filled with traffic and buses.
  0.35 a motor bike and other cars sit parked at an intersection
  0.35 An urban street with two motorcyclists, a person and a car.
  0.35 a group of motorcyclists riding with traffic in the city

 {
 "library": "faiss 1.16.0",
 "index_type": "IndexFlatIP (exact search)",
 "similarity_metric": "inner product on L2-normalised vectors (= cosine)",
 "dimension": 512,
 "image_index_vectors": 10000,
 "text_index_vectors": 50000,
 "image_index_size_mb": 20.5,
 "text_index_size_mb": 102.4,
 "indexing_time_

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>